# 01. База данных и SQL

**Цель блока:** загрузить все 9 таблиц Olist в реляционную БД (SQLite), описать связи между ними
и собрать признаки для модели **SQL-запросами с JOIN и агрегацией прямо в базе**.

Порядок работы:
1. Создаём схему с первичными и внешними ключами (`sql/01_schema.sql`) и загружаем CSV (`src/build_database.py`).
2. Проверяем целостность связей и несколько аналитических запросов с JOIN.
3. Собираем агрегаты уровня заказа (`sql/02_feature_tables.sql`) и итоговую витрину `order_features` (`sql/03_feature_mart.sql`).

### Подготовка окружения (локально или Google Colab)
В Colab ячейка подключит Google Drive и перейдёт в папку проекта: при необходимости поправьте путь `PROJECT_DIR`.
Локально (Jupyter / VS Code) ячейка просто определит пути.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/olist-capstone"   # <- папка проекта на Google Drive
    os.chdir(PROJECT_DIR)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
# В Colab базу кладём на локальный диск машины (так быстрее и надёжнее, чем на Drive).
# Каждая сессия Colab начинается с чистой машины, поэтому база при необходимости пересобирается (~1 мин).
DB_PATH = Path("/content/olist.db") if IN_COLAB else ROOT / "data" / "olist.db"
print("Папка проекта:", ROOT)
print("База данных:  ", DB_PATH)

Папка проекта: /home/claude/olist-capstone
База данных:   /home/claude/olist-capstone/data/olist.db


In [2]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd


pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Схема базы данных

Центральная таблица — `orders`. Все остальные связаны с ней напрямую или через позиции заказа:

```
 customers 1───∞ orders 1───∞ order_items ∞───1 products ∞───1 category_translation
                   │  │              │
                   │  │              └──∞───1 sellers
                   │  └──∞ order_payments
                   └─────∞ order_reviews          (здесь целевая переменная review_score)

 geolocation: связь с customers / sellers по почтовому префиксу (zip_code_prefix), много точек на префикс
```

| Таблица | Первичный ключ | Внешние ключи |
|---|---|---|
| customers | customer_id | — |
| orders | order_id | customer_id → customers |
| order_items | (order_id, order_item_id) | order_id → orders, product_id → products, seller_id → sellers |
| order_payments | (order_id, payment_sequential) | order_id → orders |
| order_reviews | (review_id, order_id) | order_id → orders |
| products | product_id | (категория → category_translation, без FK: 2 категории не переведены) |
| sellers | seller_id | — |
| category_translation | product_category_name | — |
| geolocation | — (много точек на один префикс) | логическая связь по zip_code_prefix |

Полный DDL лежит в `sql/01_schema.sql`.

In [3]:
print((ROOT / "sql" / "01_schema.sql").read_text(encoding="utf-8")[:2500], "...")

-- =====================================================================
-- Olist Brazilian E-Commerce: relational schema (SQLite)
-- 9 tables, primary keys and foreign keys reflect the real relations.
-- Центральная таблица — orders; остальные подключаются через order_id,
-- product_id, seller_id, customer_id и почтовый префикс (zip prefix).
-- =====================================================================
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS order_reviews;
DROP TABLE IF EXISTS order_payments;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS category_translation;
DROP TABLE IF EXISTS sellers;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS geolocation;

-- Клиенты. customer_id уникален для заказа, customer_unique_id — для человека.
CREATE TABLE customers (
    customer_id               TEXT PRIMARY KEY,
    customer_unique_id        TEXT NOT NULL,
    customer_zip_code_prefix  TEXT,
    customer_c

## 2. Загрузка данных

Скрипт создаёт схему, построчно вставляет 9 CSV в таблицы с ключами и сразу собирает витрину признаков.
Одна важная деталь очистки: в `customers` и `sellers` почтовые индексы записаны **без ведущих нулей**
(`9790`), а в `geolocation` — с ними (`09790`). Без приведения к 5 символам четверть клиентов «теряла» координаты.

In [4]:
from build_database import build

counts = build(DB_PATH)
con = sqlite3.connect(DB_PATH)


def q(sql: str) -> pd.DataFrame:
    """Выполнить SQL и вернуть DataFrame."""
    return pd.read_sql(sql, con)

customers                 99,441 строк
sellers                    3,095 строк
category_translation          71 строк
products                  32,951 строк
orders                    99,441 строк
order_items              112,650 строк
order_payments           103,886 строк
order_reviews             99,224 строк
geolocation            1,000,163 строк

order_features (витрина для модели): 95,824 строк

Нарушений внешних ключей: 0


In [5]:
q("""
SELECT name AS table_name, type
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

,table_name,type
0,category_translation,table
1,customers,table
2,f_geo_zip,table
3,f_items,table
4,f_main_item,table
5,f_payments,table
6,f_review,table
7,f_seller_hist,table
8,f_seller_rating,table
9,f_seller_reviews,table


`PRAGMA foreign_key_check` вернул 0 нарушений: каждый заказ ссылается на существующего клиента,
каждая позиция — на существующий товар и продавца. Схема связей корректна.

## 3. Проверка «зернистости» данных

Прежде чем соединять таблицы, важно понять, сколько строк приходится на один заказ,
иначе JOIN «размножит» заказы и исказит статистику.

In [6]:
q("""
SELECT 'order_items'    AS tbl, COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders_,
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT order_id), 3) AS rows_per_order FROM order_items
UNION ALL
SELECT 'order_payments', COUNT(*), COUNT(DISTINCT order_id),
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT order_id), 3) FROM order_payments
UNION ALL
SELECT 'order_reviews',  COUNT(*), COUNT(DISTINCT order_id),
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT order_id), 3) FROM order_reviews
""")

,tbl,rows_,orders_,rows_per_order
0,order_items,112650,98666,1.142
1,order_payments,103886,99440,1.045
2,order_reviews,99224,98673,1.006


Во всех трёх таблицах на заказ бывает больше одной строки (несколько товаров, несколько платежей,
повторные отзывы). Поэтому перед соединением с `orders` каждую из них **агрегируем до уровня заказа** (`GROUP BY order_id`).

In [7]:
q("""
SELECT order_status, COUNT(*) AS n_orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM orders
GROUP BY order_status
ORDER BY n_orders DESC
""")

,order_status,n_orders,pct
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


97% заказов имеют статус `delivered`. Для задачи берём только доставленные заказы: бизнес-вопрос
сформулирован «на момент доставки», а у отменённых и недоставленных заказов нет сроков доставки.

## 4. Аналитические запросы с JOIN

### 4.1 Опоздание доставки и оценка (orders ⋈ order_reviews)

In [8]:
q("""
SELECT CASE WHEN date(o.order_delivered_customer_date) > date(o.order_estimated_delivery_date)
            THEN 'опоздал' ELSE 'вовремя' END                 AS delivery,
       COUNT(*)                                               AS n_orders,
       ROUND(AVG(r.review_score), 2)                          AS avg_score,
       ROUND(100.0 * AVG(r.review_score <= 2), 1)             AS pct_low_1_2
FROM orders o
JOIN order_reviews r ON r.order_id = o.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery
""")

,delivery,n_orders,avg_score,pct_low_1_2
0,вовремя,89944,4.29,9.3
1,опоздал,6409,2.27,62.4


Уже простой JOIN двух таблиц показывает главный эффект проекта: при опоздании доля оценок 1–2 звезды
вырастает в несколько раз.

### 4.2 Категории товаров с наибольшей долей низких оценок (order_items ⋈ products ⋈ category_translation ⋈ order_reviews)

In [9]:
q("""
SELECT COALESCE(ct.product_category_name_english, 'unknown') AS category,
       COUNT(DISTINCT oi.order_id)                           AS n_orders,
       ROUND(AVG(r.review_score), 2)                         AS avg_score,
       ROUND(100.0 * AVG(r.review_score <= 2), 1)            AS pct_low_1_2
FROM order_items oi
JOIN products p               ON p.product_id = oi.product_id
LEFT JOIN category_translation ct ON ct.product_category_name = p.product_category_name
JOIN order_reviews r          ON r.order_id = oi.order_id
GROUP BY category
HAVING n_orders >= 1000
ORDER BY pct_low_1_2 DESC
LIMIT 10
""")

,category,n_orders,avg_score,pct_low_1_2
0,office_furniture,1263,3.49,26.1
1,unknown,1461,3.83,22.3
2,furniture_decor,6398,3.90,19.5
3,bed_bath_table,9313,3.90,19.0
4,computers_accessories,6649,3.93,18.6
5,telephony,4168,3.95,16.8
6,baby,2861,4.01,16.7
7,watches_gifts,5576,4.02,16.3
8,garden_tools,3496,4.04,16.1
9,electronics,2531,4.04,15.8


### 4.3 Продавцы: из какого штата отгрузка и насколько клиенты довольны (sellers ⋈ order_items ⋈ orders ⋈ order_reviews)

In [10]:
q("""
SELECT s.seller_state,
       COUNT(DISTINCT o.order_id)                                     AS n_orders,
       ROUND(AVG(julianday(o.order_delivered_customer_date)
                 - julianday(o.order_purchase_timestamp)), 1)         AS avg_delivery_days,
       ROUND(100.0 * AVG(r.review_score <= 2), 1)                     AS pct_low_1_2
FROM sellers s
JOIN order_items oi  ON oi.seller_id = s.seller_id
JOIN orders o        ON o.order_id = oi.order_id
JOIN order_reviews r ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_state
HAVING n_orders >= 500
ORDER BY n_orders DESC
""")

,seller_state,n_orders,avg_delivery_days,pct_low_1_2
0,SP,68173,12.2,15.4
1,MG,7683,12.7,12.6
2,PR,7461,13.4,14.0
3,RJ,4197,12.0,13.7
4,SC,3584,13.5,13.8
5,RS,1945,11.5,10.6
6,DF,807,12.5,15.0
7,BA,548,13.8,11.4


### 4.4 Число продавцов в заказе и удовлетворённость (подзапрос с агрегацией + JOIN)

In [11]:
q("""
WITH per_order AS (
    SELECT order_id, COUNT(DISTINCT seller_id) AS n_sellers, COUNT(*) AS n_items
    FROM order_items
    GROUP BY order_id
)
SELECT CASE WHEN po.n_sellers > 1 THEN '2+ продавца'
            WHEN po.n_items  > 1 THEN '1 продавец, 2+ товара'
            ELSE '1 товар' END                        AS order_type,
       COUNT(*)                                       AS n_orders,
       ROUND(AVG(r.review_score), 2)                  AS avg_score,
       ROUND(100.0 * AVG(r.review_score <= 2), 1)     AS pct_low_1_2
FROM per_order po
JOIN order_reviews r ON r.order_id = po.order_id
GROUP BY order_type
ORDER BY pct_low_1_2
""")

,order_type,n_orders,avg_score,pct_low_1_2
0,1 товар,88699,4.16,12.7
1,"1 продавец, 2+ товара",8488,3.71,24.7
2,2+ продавца,1278,2.85,47.4


Заказы из нескольких посылок от разных продавцов получают низкие оценки заметно чаще:
клиент получает заказ частями, и любая задержка одной части портит впечатление от всего заказа.

## 5. Сборка признаков в базе

Признаки собираются в два шага (всё внутри SQLite):

**Шаг 1 — `sql/02_feature_tables.sql`: агрегаты уровня заказа.**

| Таблица | Что делает | Техника SQL |
|---|---|---|
| `f_review` | последний отзыв заказа → целевая `review_score` | `ROW_NUMBER() OVER (PARTITION BY order_id ...)` |
| `f_items` | число товаров/продавцов, цена, доставка, вес, объём, фото | `JOIN products` + `GROUP BY` |
| `f_main_item` | самая дорогая позиция → категория и продавец заказа | оконная функция |
| `f_payments` | сумма, рассрочка, основной способ оплаты | `GROUP BY` + условная агрегация |
| `f_seller_hist` | сколько заказов было у продавца **до** этой покупки | `COUNT(*) OVER (... ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)` |
| `f_geo_zip` | средние координаты почтового префикса | `GROUP BY` по 1 млн точек |
| `f_seller_rating` | доля низких оценок продавца по отзывам, оставленным **до** покупки | range-JOIN по времени + `GROUP BY` |

**Шаг 2 — `sql/03_feature_mart.sql`: витрина `order_features`.** JOIN `orders`, `customers`, `sellers`,
`products`, `category_translation` и всех агрегатов; расчёт сроков доставки через `julianday()`.

**Защита от утечки данных.** В витрину попадает только то, что известно к моменту доставки.
Текст отзыва, дата создания отзыва и дата ответа в признаки **не** входят. История продавца
считается строго по событиям до даты покупки.

In [12]:
print((ROOT / "sql" / "03_feature_mart.sql").read_text(encoding="utf-8"))

-- =====================================================================
-- Финальная витрина признаков: одна строка = один доставленный заказ с отзывом.
-- JOIN 5 исходных таблиц (orders, customers, sellers, products, category_translation
-- + агрегаты f_* из 02_feature_tables.sql).
-- =====================================================================
DROP TABLE IF EXISTS order_features;
CREATE TABLE order_features AS
SELECT
    o.order_id,
    o.order_purchase_timestamp,
    -- ---------- сроки и логистика ----------
    julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)
        AS delivery_days,                         -- фактический срок доставки
    julianday(o.order_estimated_delivery_date) - julianday(o.order_purchase_timestamp)
        AS estimated_days,                        -- обещанный срок
    julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)
        AS delay_days,                            -- >0 = о

In [13]:
features = q("SELECT * FROM order_features")
print(features.shape)
features.head()

(95824, 41)


,order_id,order_purchase_timestamp,delivery_days,estimated_days,delay_days,is_late,approval_hours,seller_handling_days,carrier_days,seller_missed_limit,purchase_month,purchase_dow,purchase_hour,n_items,n_products,n_sellers,items_price,freight_value,max_item_price,freight_ratio,total_weight_g,total_volume_cm3,avg_photos_qty,avg_description_len,category,payment_value,max_installments,n_payments,n_payment_types,payment_type,customer_state,seller_state,same_state,customer_lat,customer_lng,seller_lat,seller_lng,seller_prior_orders,seller_prior_reviews,seller_prior_low_rate,review_score
0,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,54.813194,18.488449,36.324745,1,0.000000,53.205035,1.608160,1,9,4,12,3,1,1,134.97,8.49,44.99,0.062903,3000.0,12288.0,1.0,1036.0,health_beauty,NaN,NaN,NaN,NaN,NaN,SP,PR,0,-20.585751,-47.863693,-25.507014,-49.275963,0,0,NaN,1
1,3b697a20d9e427646d92567910af6d57,2016-10-03 09:44:50,23.178738,23.593866,-0.415127,0,78.101111,16.924525,3.000000,1,10,1,9,1,1,1,29.90,15.56,29.90,0.520401,300.0,4096.0,3.0,1642.0,watches_gifts,45.46,1.0,1.0,1.0,boleto,SP,PR,0,-23.581451,-46.635029,-24.959184,-53.462644,0,0,NaN,4
2,be5bc2f0da14d8071e2d45451ad119d9,2016-10-03 16:56:50,24.057500,34.293866,-10.236366,0,71.115000,15.020856,6.073519,0,10,1,16,1,1,1,21.90,17.19,21.90,0.784932,400.0,4096.0,1.0,518.0,sports_leisure,39.09,1.0,1.0,1.0,boleto,RS,SP,0,-28.293541,-53.502238,-21.143389,-48.995314,0,0,NaN,4
3,a41c8759fbe7aab36ea07e038b2d4465,2016-10-03 21:13:36,30.572581,56.115556,-25.542975,0,29.970278,20.365394,8.958426,1,10,1,21,1,1,1,36.49,17.24,36.49,0.472458,767.0,4160.0,1.0,141.0,sports_leisure,53.73,1.0,1.0,1.0,boleto,RS,SP,0,-30.041161,-51.213661,-23.502755,-47.430451,0,0,NaN,3
4,d207cc272675637bfed0062edffd0818,2016-10-03 22:06:03,27.542813,50.079132,-22.536319,0,12.367778,17.163542,9.863947,0,10,1,22,1,1,1,119.90,13.56,119.90,0.113094,2050.0,14960.0,1.0,130.0,furniture_decor,133.46,6.0,1.0,1.0,credit_card,SP,SP,1,-22.892792,-47.173849,-21.757321,-48.829744,0,0,NaN,1


In [14]:
q("""
SELECT COUNT(*)                                  AS n_orders,
       ROUND(AVG(review_score), 3)               AS avg_score,
       ROUND(100.0 * AVG(review_score <= 2), 2)  AS pct_low_1_2,
       MIN(order_purchase_timestamp)             AS first_purchase,
       MAX(order_purchase_timestamp)             AS last_purchase
FROM order_features
""")

,n_orders,avg_score,pct_low_1_2,first_purchase,last_purchase
0,95824,4.156,12.81,2016-09-15 12:16:38,2018-08-29 15:00:37


### Проверка витрины
* одна строка = один заказ (дубликатов `order_id` нет);
* в выборку вошли доставленные заказы с отзывом;
* пропуски есть только в отдельных признаках (фото товара, координаты, история продавца) — их разбор в ноутбуке EDA.

In [15]:
print("Дубликатов order_id:", features["order_id"].duplicated().sum())
features.isna().sum().loc[lambda s: s > 0].to_frame("n_missing")

Дубликатов order_id: 0


,n_missing
approval_hours,14
seller_handling_days,15
carrier_days,1
total_weight_g,16
total_volume_cm3,16
avg_photos_qty,1324
avg_description_len,1324
payment_value,1
max_installments,1
n_payments,1


In [16]:
con.close()